# Wikidata Person Identifier Discovery

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook systematically discovers missing external identifiers for a person already in Wikidata. Starting from a Wikidata QID, it:

1. **Retrieves existing identifiers** from the person's Wikidata item
2. **Searches external platforms** using name, ORCID, and other known identifiers to find missing profiles
3. **Checks national library authority systems** for catalog records that could yield new authority IDs
4. **Generates QuickStatements** to add verified missing identifiers
5. **Creates Google Alerts queries** for monitoring when new authority records appear

## Key Features

- **Comprehensive Coverage**: Checks 40+ external identifier platforms
- **Smart Detection**: Uses ORCID, name variants, and existing IDs
- **National Library Focus**: GND, NLI, Trove, LIBRIS, NUKAT, etc.
- **Verification Links**: Direct URLs to verify discovered profiles
- **Google Alerts Generator**: Site-scoped alert queries
- **QuickStatements Output**: Ready-to-use statements
- **Benchmark Comparison**: Compare against top anthropologists

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

In [1]:
# Install required packages
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import time
import re
import json
from datetime import datetime
from typing import Dict, List, Optional, Any
from urllib.parse import quote, quote_plus
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print('Setup complete.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.9 MB/s eta 0:00:00
Setup complete.


## Configuration

In [2]:
class Config:
    WIKIDATA_ENDPOINT = 'https://query.wikidata.org/sparql'
    REQUEST_DELAY = 1.0
    TIMEOUT = 30
    USER_AGENT = 'WikidataIdentifierDiscovery/1.0 (https://www.mattartz.me/)'

config = Config()
print(f'Configuration loaded. Endpoint: {config.WIKIDATA_ENDPOINT}')

Configuration loaded. Endpoint: https://query.wikidata.org/sparql


## Identifier Property Definitions

In [3]:
IDENTIFIER_PROPERTIES = {
    'library_authority': {
        'P214': {'name': 'VIAF cluster ID', 'url': 'https://viaf.org/viaf/{id}'},
        'P213': {'name': 'ISNI', 'url': 'https://isni.org/isni/{id}'},
        'P244': {'name': 'Library of Congress authority ID', 'url': 'https://id.loc.gov/authorities/names/{id}'},
        'P227': {'name': 'GND ID', 'url': 'https://d-nb.info/gnd/{id}'},
        'P268': {'name': 'BnF ID (France)', 'url': 'https://catalogue.bnf.fr/ark:/12148/cb{id}'},
        'P269': {'name': 'IdRef ID (France)', 'url': 'https://www.idref.fr/{id}'},
        'P8189': {'name': 'National Library of Israel J9U ID', 'url': 'https://www.nli.org.il/en/authorities/{id}'},
        'P1315': {'name': 'NLA Trove people ID', 'url': 'https://trove.nla.gov.au/people/{id}'},
        'P5587': {'name': 'Libris-URI (Sweden)', 'url': 'https://libris.kb.se/{id}'},
        'P1207': {'name': 'NUKAT ID (Poland)', 'url': 'https://katalog.nukat.edu.pl/search.xhtml?p={id}'},
        'P396': {'name': 'SBN author ID (Italy)', 'url': 'https://opac.sbn.it/risultati-autori/-/opac-autori/{id}'},
        'P950': {'name': 'BNE ID (Spain)', 'url': 'https://datos.bne.es/resource/{id}'},
        'P349': {'name': 'NDL Authority ID (Japan)', 'url': 'https://id.ndl.go.jp/auth/ndlna/{id}'},
        'P271': {'name': 'CiNii author ID (Japan)', 'url': 'https://ci.nii.ac.jp/author/{id}'},
        'P5034': {'name': 'National Library of Korea ID', 'url': 'https://lod.nl.go.kr/resource/{id}'},
        'P1006': {'name': 'NTA ID (Netherlands)', 'url': 'http://data.bibliotheken.nl/id/thes/p{id}'},
        'P7859': {'name': 'WorldCat Identities ID', 'url': 'https://www.worldcat.org/identities/{id}'},
    },
    'academic_research': {
        'P496': {'name': 'ORCID iD', 'url': 'https://orcid.org/{id}'},
        'P4012': {'name': 'Semantic Scholar author ID', 'url': 'https://www.semanticscholar.org/author/{id}'},
        'P10283': {'name': 'OpenAlex ID', 'url': 'https://openalex.org/{id}'},
        'P2798': {'name': 'Loop ID (Frontiers)', 'url': 'https://loop.frontiersin.org/people/{id}/overview'},
        'P1153': {'name': 'Scopus author ID', 'url': 'https://www.scopus.com/authid/detail.uri?authorId={id}'},
        'P1960': {'name': 'Google Scholar author ID', 'url': 'https://scholar.google.com/citations?user={id}'},
        'P3829': {'name': 'Publons author ID', 'url': 'https://publons.com/researcher/{id}/'},
        'P1053': {'name': 'ResearcherID', 'url': 'https://www.webofscience.com/wos/author/record/{id}'},
        'P5237': {'name': 'Dimensions author ID', 'url': 'https://app.dimensions.ai/discover/publication?and_facet_researcher={id}'},
    },
    'academic_social': {
        'P2038': {'name': 'ResearchGate profile ID', 'url': 'https://www.researchgate.net/profile/{id}'},
        'P5715': {'name': 'Academia.edu profile URL', 'url': '{id}', 'is_url': True},
        'P11780': {'name': 'Humanities Commons member ID', 'url': 'https://hcommons.org/members/{id}/'},
        'P10557': {'name': 'Zotero ID', 'url': 'https://www.zotero.org/{id}'},
    },
    'music_creative': {
        'P434': {'name': 'MusicBrainz artist ID', 'url': 'https://musicbrainz.org/artist/{id}'},
        'P1953': {'name': 'Discogs artist ID', 'url': 'https://www.discogs.com/artist/{id}'},
        'P1902': {'name': 'Spotify artist ID', 'url': 'https://open.spotify.com/artist/{id}'},
        'P3192': {'name': 'Last.fm ID', 'url': 'https://www.last.fm/music/{id}'},
        'P3040': {'name': 'SoundCloud ID', 'url': 'https://soundcloud.com/{id}'},
        'P5917': {'name': 'Shazam artist ID', 'url': 'https://www.shazam.com/artist/{id}'},
        'P3283': {'name': 'Bandcamp ID', 'url': 'https://{id}.bandcamp.com'},
        'P2850': {'name': 'iTunes artist ID', 'url': 'https://music.apple.com/artist/{id}'},
    },
    'podcast': {
        'P5381': {'name': 'Podchaser creator ID', 'url': 'https://www.podchaser.com/creators/{id}'},
    },
    'books_publishing': {
        'P648': {'name': 'Open Library author ID', 'url': 'https://openlibrary.org/authors/{id}'},
        'P2963': {'name': 'Goodreads author ID', 'url': 'https://www.goodreads.com/author/show/{id}'},
        'P4862': {'name': 'Amazon author ID', 'url': 'https://www.amazon.com/author/{id}'},
        'P5408': {'name': 'BookBrainz author ID', 'url': 'https://bookbrainz.org/author/{id}'},
    },
    'general_social': {
        'P6634': {'name': 'LinkedIn personal profile ID', 'url': 'https://www.linkedin.com/in/{id}'},
        'P2002': {'name': 'Twitter/X username', 'url': 'https://twitter.com/{id}'},
        'P2037': {'name': 'GitHub username', 'url': 'https://github.com/{id}'},
        'P2397': {'name': 'YouTube channel ID', 'url': 'https://www.youtube.com/channel/{id}'},
        'P345': {'name': 'IMDb ID', 'url': 'https://www.imdb.com/name/{id}'},
        'P7879': {'name': 'Medium username', 'url': 'https://medium.com/@{id}'},
        'P6479': {'name': 'Unsplash username', 'url': 'https://unsplash.com/@{id}'},
    },
}

# Flatten for lookup
ALL_PROPERTIES = {}
for cat, props in IDENTIFIER_PROPERTIES.items():
    for pid, info in props.items():
        info['category'] = cat
        ALL_PROPERTIES[pid] = info

print(f'Loaded {len(ALL_PROPERTIES)} identifier properties across {len(IDENTIFIER_PROPERTIES)} categories')

Loaded 50 identifier properties across 7 categories


## National Library Search Sites (for Google Alerts)

In [4]:
NATIONAL_LIBRARY_SITES = {
    'VIAF': {'site': 'viaf.org', 'prop': 'P214', 'name': 'Virtual International Authority File'},
    'ISNI': {'site': 'isni.org', 'prop': 'P213', 'name': 'International Standard Name Identifier'},
    'WorldCat': {'site': 'worldcat.org', 'prop': 'P7859', 'name': 'WorldCat'},
    'LCNAF': {'site': 'id.loc.gov', 'prop': 'P244', 'name': 'Library of Congress'},
    'GND': {'site': 'd-nb.info', 'prop': 'P227', 'name': 'German National Library'},
    'GND_LOBID': {'site': 'lobid.org', 'prop': 'P227', 'name': 'GND via lobid.org'},
    'BnF': {'site': 'bnf.fr', 'prop': 'P268', 'name': 'Bibliotheque nationale de France'},
    'IdRef': {'site': 'idref.fr', 'prop': 'P269', 'name': 'IdRef (France)'},
    'NTA': {'site': 'data.bibliotheken.nl', 'prop': 'P1006', 'name': 'Netherlands NTA'},
    'LIBRIS': {'site': 'libris.kb.se', 'prop': 'P5587', 'name': 'Sweden LIBRIS'},
    'NUKAT': {'site': 'nukat.edu.pl', 'prop': 'P1207', 'name': 'Poland NUKAT'},
    'SBN': {'site': 'opac.sbn.it', 'prop': 'P396', 'name': 'Italy SBN'},
    'BNE': {'site': 'bne.es', 'prop': 'P950', 'name': 'Spain BNE'},
    'Trove': {'site': 'trove.nla.gov.au', 'prop': 'P1315', 'name': 'Australia Trove'},
    'NDL': {'site': 'id.ndl.go.jp', 'prop': 'P349', 'name': 'Japan National Diet Library'},
    'CiNii': {'site': 'ci.nii.ac.jp', 'prop': 'P271', 'name': 'Japan CiNii'},
    'NLI': {'site': 'nli.org.il', 'prop': 'P8189', 'name': 'National Library of Israel'},
    'NLK': {'site': 'lod.nl.go.kr', 'prop': 'P5034', 'name': 'National Library of Korea'},
}

print(f'Configured {len(NATIONAL_LIBRARY_SITES)} national library search sites')

Configured 18 national library search sites


## Wikidata Query Functions

In [5]:
def query_wikidata(sparql: str):    """Execute SPARQL query against Wikidata."""    try:        r = requests.get(            config.WIKIDATA_ENDPOINT,            params={'query': sparql, 'format': 'json'},            headers={'User-Agent': config.USER_AGENT},            timeout=config.TIMEOUT        )        r.raise_for_status()        return r.json()    except Exception as e:        print(f"Query error: {e}")        return Nonedef get_person_info(qid: str):    """Get basic info and all external identifiers for a person."""    qid = qid.upper().strip()    if not qid.startswith('Q'):        qid = 'Q' + qid        # Basic info query    basic_q = f"""    SELECT ?label ?desc ?birthYear WHERE {{      wd:{qid} rdfs:label ?label FILTER(LANG(?label) = "en").      OPTIONAL {{ wd:{qid} schema:description ?desc FILTER(LANG(?desc) = "en"). }}      OPTIONAL {{ wd:{qid} wdt:P569 ?birth. BIND(YEAR(?birth) AS ?birthYear) }}    }} LIMIT 1    """        result = query_wikidata(basic_q)    if not result or not result.get('results', {}).get('bindings'):        return None        b = result['results']['bindings'][0]    info = {        'qid': qid,        'label': b.get('label', {}).get('value', ''),        'description': b.get('desc', {}).get('value', ''),        'birth_year': b.get('birthYear', {}).get('value', ''),        'identifiers': {}    }        # Get identifiers in batches    props = list(ALL_PROPERTIES.keys())    batch_size = 25        for i in range(0, len(props), batch_size):        batch = props[i:i+batch_size]        optionals = "\n".join([f"OPTIONAL {{ wd:{qid} wdt:{p} ?{p}. }}" for p in batch])        selects = " ".join([f"?{p}" for p in batch])                id_q = f"SELECT {selects} WHERE {{ {optionals} }}"        result = query_wikidata(id_q)                if result and result.get('results', {}).get('bindings'):            for binding in result['results']['bindings']:                for p in batch:                    if p in binding and binding[p].get('value'):                        val = binding[p]['value']                        if p not in info['identifiers']:                            info['identifiers'][p] = []                        if val not in info['identifiers'][p]:                            info['identifiers'][p].append(val)        time.sleep(0.3)        return infoprint("Query functions defined.")

SyntaxError: invalid syntax (ipython-input-796084334.py, line 1)

## External Platform Search Functions

In [ ]:
def search_semantic_scholar(orcid=None, name=None):    """Search Semantic Scholar by ORCID or name."""    headers = {'User-Agent': config.USER_AGENT}    try:        if orcid:            url = f"https://api.semanticscholar.org/graph/v1/author/orcid:{orcid}"            params = {'fields': 'authorId,name,affiliations,paperCount,citationCount,externalIds'}        elif name:            url = "https://api.semanticscholar.org/graph/v1/author/search"            params = {'query': name, 'fields': 'authorId,name,affiliations,paperCount,externalIds', 'limit': 5}        else:            return None                r = requests.get(url, params=params, headers=headers, timeout=config.TIMEOUT)        if r.status_code == 200:            data = r.json()            return data['data'][0] if 'data' in data and data['data'] else data    except Exception as e:        print(f"Semantic Scholar error: {e}")    return Nonedef search_openalex(orcid=None, name=None):    """Search OpenAlex by ORCID or name."""    headers = {'User-Agent': config.USER_AGENT}    try:        if orcid:            url = f"https://api.openalex.org/authors/orcid:{orcid}"        elif name:            url = f"https://api.openalex.org/authors?filter=display_name.search:{quote(name)}"        else:            return None                r = requests.get(url, headers=headers, timeout=config.TIMEOUT)        if r.status_code == 200:            data = r.json()            return data['results'][0] if 'results' in data and data['results'] else data    except Exception as e:        print(f"OpenAlex error: {e}")    return Nonedef search_orcid_external_ids(orcid: str):    """Get external IDs from ORCID record."""    headers = {'Accept': 'application/json', 'User-Agent': config.USER_AGENT}    try:        r = requests.get(f"https://pub.orcid.org/v3.0/{orcid}/external-identifiers",                         headers=headers, timeout=config.TIMEOUT)        if r.status_code == 200:            return r.json()    except Exception as e:        print(f"ORCID error: {e}")    return Nonedef search_musicbrainz(name: str):    """Search MusicBrainz for artist."""    headers = {'User-Agent': config.USER_AGENT, 'Accept': 'application/json'}    try:        url = f"https://musicbrainz.org/ws/2/artist/?query=artist:{quote(name)}&fmt=json&limit=5"        r = requests.get(url, headers=headers, timeout=config.TIMEOUT)        if r.status_code == 200:            return r.json().get('artists', [])    except Exception as e:        print(f"MusicBrainz error: {e}")    return Nonedef search_open_library(name: str):    """Search Open Library for author."""    headers = {'User-Agent': config.USER_AGENT}    try:        url = f"https://openlibrary.org/search/authors.json?q={quote(name)}&limit=5"        r = requests.get(url, headers=headers, timeout=config.TIMEOUT)        if r.status_code == 200:            return r.json().get('docs', [])    except Exception as e:        print(f"Open Library error: {e}")    return Noneprint("External search functions defined.")

## Discovery Engine

In [ ]:
def discover_missing_identifiers(person_info, progress_fn=None):    """Search external platforms for missing identifiers."""    results = {        'existing': {},        'discovered': {},        'needs_verification': {},        'errors': []    }        name = person_info['label']    orcid = person_info['identifiers'].get('P496', [None])[0]    existing = person_info['identifiers']        # Categorize existing    for pid, values in existing.items():        if pid in ALL_PROPERTIES:            prop = ALL_PROPERTIES[pid]            results['existing'][pid] = {                'name': prop['name'],                'values': values,                'urls': [prop['url'].format(id=v) for v in values]            }        steps = 5    step = 0        def progress(msg):        nonlocal step        step += 1        if progress_fn:            progress_fn(step, steps, msg)        # 1. Semantic Scholar    progress("Searching Semantic Scholar...")    if 'P4012' not in existing:        ss = search_semantic_scholar(orcid=orcid) if orcid else search_semantic_scholar(name=name)        time.sleep(config.REQUEST_DELAY)        if ss and ss.get('authorId'):            results['discovered']['P4012'] = {                'name': 'Semantic Scholar author ID',                'value': ss['authorId'],                'url': f"https://www.semanticscholar.org/author/{ss['authorId']}",                'confidence': 'high' if orcid else 'medium',                'matched_name': ss.get('name', ''),                'info': f"Papers: {ss.get('paperCount', '?')}"            }        # 2. OpenAlex    progress("Searching OpenAlex...")    if 'P10283' not in existing:        oa = search_openalex(orcid=orcid) if orcid else search_openalex(name=name)        time.sleep(config.REQUEST_DELAY)        if oa and oa.get('id'):            oa_id = oa['id'].replace('https://openalex.org/', '')            results['discovered']['P10283'] = {                'name': 'OpenAlex ID',                'value': oa_id,                'url': oa['id'],                'confidence': 'high' if orcid else 'medium',                'matched_name': oa.get('display_name', ''),                'info': f"Works: {oa.get('works_count', '?')}"            }        # 3. ORCID external IDs    progress("Checking ORCID external IDs...")    if orcid:        orcid_ext = search_orcid_external_ids(orcid)        time.sleep(config.REQUEST_DELAY)        if orcid_ext and orcid_ext.get('external-identifier'):            mapping = {                'Scopus Author ID': 'P1153',                'ResearcherID': 'P1053',                'Loop profile': 'P2798',                'ISNI': 'P213',                'GitHub': 'P2037',                'Google Scholar': 'P1960',                'ResearchGate': 'P2038',            }            for ext in orcid_ext.get('external-identifier', []):                id_type = ext.get('external-id-type', '')                id_val = ext.get('external-id-value', '')                if id_type in mapping:                    pid = mapping[id_type]                    if pid not in existing and pid not in results['discovered']:                        prop = ALL_PROPERTIES.get(pid, {})                        results['discovered'][pid] = {                            'name': prop.get('name', id_type),                            'value': id_val,                            'url': prop.get('url', '').format(id=id_val),                            'confidence': 'high',                            'source': 'ORCID'                        }        # 4. MusicBrainz    progress("Searching MusicBrainz...")    if 'P434' not in existing:        mb = search_musicbrainz(name)        time.sleep(config.REQUEST_DELAY)        if mb:            for artist in mb:                if artist.get('name', '').lower() == name.lower():                    results['discovered']['P434'] = {                        'name': 'MusicBrainz artist ID',                        'value': artist.get('id'),                        'url': f"https://musicbrainz.org/artist/{artist.get('id')}",                        'confidence': 'medium',                        'matched_name': artist.get('name', '')                    }                    break        # 5. Open Library    progress("Searching Open Library...")    if 'P648' not in existing:        ol = search_open_library(name)        time.sleep(config.REQUEST_DELAY)        if ol:            for author in ol:                if author.get('name', '').lower() == name.lower():                    ol_key = author.get('key', '').replace('/authors/', '')                    results['discovered']['P648'] = {                        'name': 'Open Library author ID',                        'value': ol_key,                        'url': f"https://openlibrary.org/authors/{ol_key}",                        'confidence': 'medium',                        'matched_name': author.get('name', ''),                        'info': f"Works: {author.get('work_count', '?')}"                    }                    break            else:                if ol:                    results['needs_verification']['P648'] = {                        'name': 'Open Library author ID',                        'candidates': [{'value': a.get('key', '').replace('/authors/', ''),                                       'name': a.get('name', ''),                                       'work_count': a.get('work_count', 0)} for a in ol[:3]]                    }        return resultsprint("Discovery engine defined.")

## Google Alerts Generator

In [ ]:
def generate_google_alerts(person_info, existing_ids, book_titles=None, isbns=None):    """Generate Google Alert queries for national library monitoring."""    name = person_info['label']    birth_year = person_info.get('birth_year', '')        # Name variants    parts = name.split()    if len(parts) >= 2:        first, last = parts[0], parts[-1]        inverted = f"{last}, {first}"        inverted_year = f"{last}, {first}, {birth_year}" if birth_year else None    else:        inverted = name        inverted_year = None        alerts = {        'high_priority': [],        'medium_priority': [],        'book_alerts': [],        'isbn_alerts': [],        'authority_alerts': [            f'"{inverted}" "authority"',            f'"{name}" "authority file"',            f'"{inverted}" "Normdaten"',        ]    }        high_priority_libs = ['VIAF', 'GND', 'BnF', 'LCNAF', 'Trove', 'NDL', 'LIBRIS']        for lib_key, lib_info in NATIONAL_LIBRARY_SITES.items():        prop = lib_info['prop']        site = lib_info['site']                if prop not in existing_ids:            alert = {                'library': lib_info['name'],                'site': site,                'prop': prop,                'queries': [                    f'"{name}" site:{site}',                    f'"{inverted}" site:{site}'                ]            }            if inverted_year:                alert['queries'].append(f'"{inverted_year}" site:{site}')                        if lib_key in high_priority_libs:                alerts['high_priority'].append(alert)            else:                alerts['medium_priority'].append(alert)        # Book alerts    if book_titles:        for title in book_titles:            for lib_key in ['GND', 'LIBRIS', 'Trove', 'NUKAT', 'BnF', 'NDL']:                if lib_key in NATIONAL_LIBRARY_SITES:                    lib = NATIONAL_LIBRARY_SITES[lib_key]                    if lib['prop'] not in existing_ids:                        alerts['book_alerts'].append({                            'title': title,                            'library': lib['name'],                            'query': f'"{title}" site:{lib["site"]}'                        })        # ISBN alerts    if isbns:        for isbn in isbns:            for lib_key in ['GND', 'LIBRIS', 'Trove', 'NUKAT', 'BnF', 'NDL']:                if lib_key in NATIONAL_LIBRARY_SITES:                    lib = NATIONAL_LIBRARY_SITES[lib_key]                    if lib['prop'] not in existing_ids:                        alerts['isbn_alerts'].append({                            'isbn': isbn,                            'library': lib['name'],                            'query': f'{isbn} site:{lib["site"]}'                        })        return alertsprint("Google Alerts generator defined.")

## Manual Check URL Generator

In [ ]:
def generate_manual_check_urls(name, existing_ids):    """Generate URLs for manual platform checking."""    encoded = quote_plus(name)        checks = {        'academic': [            {'name': 'Humanities Commons', 'pid': 'P11780', 'url': f'https://hcommons.org/?s={encoded}'},            {'name': 'Zotero', 'pid': 'P10557', 'url': f'https://www.zotero.org/search/#q={encoded}'},            {'name': 'Academia.edu', 'pid': 'P5715', 'url': f'https://www.academia.edu/people/search?q={encoded}'},            {'name': 'ResearchGate', 'pid': 'P2038', 'url': f'https://www.researchgate.net/search/researcher?q={encoded}'},            {'name': 'Loop (Frontiers)', 'pid': 'P2798', 'url': f'https://loop.frontiersin.org/search?search={encoded}'},            {'name': 'Google Scholar', 'pid': 'P1960', 'url': f'https://scholar.google.com/citations?view_op=search_authors&mauthors={encoded}'},        ],        'music': [            {'name': 'Last.fm', 'pid': 'P3192', 'url': f'https://www.last.fm/search/artists?q={encoded}'},            {'name': 'SoundCloud', 'pid': 'P3040', 'url': f'https://soundcloud.com/search/people?q={encoded}'},            {'name': 'Bandcamp', 'pid': 'P3283', 'url': f'https://bandcamp.com/search?q={encoded}&item_type=b'},            {'name': 'Spotify', 'pid': 'P1902', 'url': f'https://open.spotify.com/search/{encoded}'},            {'name': 'Shazam', 'pid': 'P5917', 'url': f'https://www.shazam.com/search/{encoded}'},        ],        'social': [            {'name': 'LinkedIn', 'pid': 'P6634', 'url': f'https://www.linkedin.com/search/results/people/?keywords={encoded}'},            {'name': 'GitHub', 'pid': 'P2037', 'url': f'https://github.com/search?q={encoded}&type=users'},            {'name': 'Medium', 'pid': 'P7879', 'url': f'https://medium.com/search?q={encoded}'},            {'name': 'Podchaser', 'pid': 'P5381', 'url': f'https://www.podchaser.com/search?term={encoded}'},        ],        'books': [            {'name': 'Goodreads', 'pid': 'P2963', 'url': f'https://www.goodreads.com/search?q={encoded}&search_type=authors'},            {'name': 'Open Library', 'pid': 'P648', 'url': f'https://openlibrary.org/search/authors?q={encoded}'},            {'name': 'Amazon Author', 'pid': 'P4862', 'url': f'https://www.amazon.com/s?k={encoded}&i=stripbooks'},        ],    }        # Filter out existing    filtered = {}    for cat, items in checks.items():        filtered[cat] = [i for i in items if i['pid'] not in existing_ids]        return filtereddef generate_quickstatements(qid, discovered):    """Generate QuickStatements for discovered identifiers."""    lines = []    for pid, info in discovered.items():        val = info.get('value', '')        if val:            lines.append(f'{qid}\t{pid}\t"{val}"')    return "\n".join(lines)print("Utility functions defined.")

---

## Interactive Discovery Interface

In [ ]:
# Widgetsqid_input = widgets.Text(value='Q105757768', description='Wikidata QID:',                          layout=widgets.Layout(width='400px'))book_titles_input = widgets.Textarea(placeholder='Book titles (one per line)',                                     description='Books:', layout=widgets.Layout(width='500px', height='80px'))isbn_input = widgets.Textarea(placeholder='ISBNs (one per line)',                              description='ISBNs:', layout=widgets.Layout(width='500px', height='60px'))run_btn = widgets.Button(description='Discover Identifiers', button_style='primary')progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:')status = widgets.Label(value='')output = widgets.Output()# Global results storageglobal_results = {}def run_discovery(b):    global global_results    with output:        clear_output()        qid = qid_input.value.strip()        if not qid:            print("Please enter a QID")            return                progress.value = 0        status.value = 'Fetching person info...'                person = get_person_info(qid)        if not person:            print(f"Could not find: {qid}")            return                progress.value = 10                print("=" * 70)        print("WIKIDATA IDENTIFIER DISCOVERY REPORT")        print("=" * 70)        print(f"\nPerson: {person['label']}")        print(f"QID: {person['qid']}")        print(f"Description: {person.get('description', 'N/A')}")        print(f"URL: https://www.wikidata.org/wiki/{person['qid']}")                print(f"\nExisting identifiers: {len(person['identifiers'])}")        for cat, props in IDENTIFIER_PROPERTIES.items():            count = sum(1 for p in props if p in person['identifiers'])            print(f"  {cat}: {count}/{len(props)}")                def update_progress(step, total, msg):            progress.value = 10 + int((step/total) * 60)            status.value = msg                print("\n" + "-" * 70)        print("SEARCHING EXTERNAL PLATFORMS...")        print("-" * 70)                results = discover_missing_identifiers(person, update_progress)        progress.value = 70                # Show discovered        print(f"\nDISCOVERED ({len(results['discovered'])})")        print("-" * 40)        if results['discovered']:            for pid, info in results['discovered'].items():                conf = info.get('confidence', '?')                icon = 'HIGH' if conf == 'high' else 'MED' if conf == 'medium' else 'LOW'                print(f"\n  [{icon}] {pid} {info['name']}")                print(f"      Value: {info['value']}")                print(f"      URL: {info.get('url', 'N/A')}")                if info.get('matched_name'):                    print(f"      Matched: {info['matched_name']}")        else:            print("  None found automatically.")                progress.value = 80                # Manual checks        manual = generate_manual_check_urls(person['label'], person['identifiers'])        print(f"\nMANUAL CHECK URLS")        print("-" * 40)        for cat, items in manual.items():            if items:                print(f"\n  {cat.upper()}:")                for item in items:                    print(f"    {item['name']}: {item['url']}")                progress.value = 90                # Google Alerts        books = [t.strip() for t in book_titles_input.value.split('\n') if t.strip()]        isbns = [i.strip() for i in isbn_input.value.split('\n') if i.strip()]                alerts = generate_google_alerts(person, person['identifiers'], books, isbns)                print(f"\nGOOGLE ALERTS FOR NATIONAL LIBRARIES")        print("-" * 40)        print("\nHIGH PRIORITY:")        for alert in alerts['high_priority']:            print(f"\n  {alert['library']} ({alert['prop']})")            for q in alert['queries']:                print(f"    {q}")                if alerts['medium_priority']:            print("\n\nMEDIUM PRIORITY:")            for alert in alerts['medium_priority']:                print(f"\n  {alert['library']} ({alert['prop']})")                print(f"    {alert['queries'][0]}")                print("\n\nGENERAL AUTHORITY ALERTS:")        for q in alerts['authority_alerts']:            print(f"  {q}")                # QuickStatements        if results['discovered']:            qs = generate_quickstatements(person['qid'], results['discovered'])            print(f"\nQUICKSTATEMENTS")            print("-" * 40)            print(qs)                # Store for export        global_results = {            'person': person,            'results': results,            'manual': manual,            'alerts': alerts,            'timestamp': datetime.now().isoformat()        }                progress.value = 100        status.value = 'Complete!'                print("\n" + "=" * 70)        print("SUMMARY")        print("=" * 70)        print(f"Existing: {len(results['existing'])}")        print(f"Discovered: {len(results['discovered'])}")        print(f"Needs verification: {len(results['needs_verification'])}")        print(f"Manual checks: {sum(len(v) for v in manual.values())}")        print(f"Google Alerts: {len(alerts['high_priority']) + len(alerts['medium_priority'])} libraries")run_btn.on_click(run_discovery)display(widgets.VBox([    widgets.HTML('<h3>Wikidata Person Identifier Discovery</h3>'),    qid_input,    widgets.HTML('<br><b>Optional: Books to monitor</b>'),    book_titles_input,    isbn_input,    run_btn,    progress,    status,    output]))

## Export Results

In [ ]:
if global_results:    ts = datetime.now().strftime('%Y%m%d_%H%M%S')    qid = global_results['person']['qid']        # Discovered CSV    discovered = global_results['results'].get('discovered', {})    if discovered:        rows = [{'property': p, 'name': i.get('name'), 'value': i.get('value'),                 'url': i.get('url'), 'confidence': i.get('confidence')}                for p, i in discovered.items()]        df = pd.DataFrame(rows)        fn = f"discovered_{qid}_{ts}.csv"        df.to_csv(fn, index=False)        print(f"Saved: {fn}")        # Google Alerts    alerts_fn = f"google_alerts_{qid}_{ts}.txt"    with open(alerts_fn, 'w') as f:        f.write(f"Google Alerts for {global_results['person']['label']}\n")        f.write("=" * 50 + "\n\n")        for a in global_results['alerts']['high_priority']:            f.write(f"\n{a['library']} ({a['prop']}):\n")            for q in a['queries']:                f.write(f"  {q}\n")        for a in global_results['alerts']['medium_priority']:            f.write(f"\n{a['library']} ({a['prop']}):\n")            for q in a['queries']:                f.write(f"  {q}\n")    print(f"Saved: {alerts_fn}")        # QuickStatements    if discovered:        qs = generate_quickstatements(qid, discovered)        qs_fn = f"quickstatements_{qid}_{ts}.txt"        with open(qs_fn, 'w') as f:            f.write(qs)        print(f"Saved: {qs_fn}")        # Full JSON    json_fn = f"discovery_{qid}_{ts}.json"    export = {        'person': {'qid': qid, 'label': global_results['person']['label']},        'existing': list(global_results['results']['existing'].keys()),        'discovered': global_results['results']['discovered'],        'needs_verification': global_results['results']['needs_verification'],        'alerts': global_results['alerts'],        'timestamp': global_results['timestamp']    }    with open(json_fn, 'w') as f:        json.dump(export, f, indent=2)    print(f"Saved: {json_fn}")else:    print("Run discovery first.")

In [ ]:
# Download files
try:
    from google.colab import files
    import glob
    if global_results:
        qid = global_results['person']['qid']
        for f in glob.glob(f"*{qid}*"):
            files.download(f)
            print(f"Downloaded: {f}")
except ImportError:
    print("Files saved to working directory.")

---

## Reference: All Identifier Properties

In [ ]:
print("COMPLETE IDENTIFIER PROPERTY REFERENCE")print("=" * 70)for category, props in IDENTIFIER_PROPERTIES.items():    print(f"\n{category.upper().replace('_', ' ')} ({len(props)} properties)")    print("-" * 50)    for pid, info in props.items():        print(f"  {pid:8} {info['name']}")        print(f"           {info['url']}")